In [3]:
# Install Dependencies
!pip install torch torchvision transformers
# Verify GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

  Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl (2.4 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
CUDA Available: True
GPU: NVIDIA GeForce RTX 3060



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Import Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg19
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
from swin_transformer import SwinTransformerSys

# Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


D:\User\Jansen\Self Study\2025 - 05 - MAY\LiPAD\lipad-venv\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [9]:
# Swin UNet Generator with Text Reconstruction Module
class SwinUNet(nn.Module):
    def __init__(self, img_size=(128, 256), patch_size=4, in_chans=3, out_chans=3):
        super(SwinUNet, self).__init__()
        # Swin Transformer encoder (Swin-T backbone)
        self.swin = SwinTransformerModel.from_pretrained(
            "microsoft/swin-tiny-patch4-window7-224",
            image_size=img_size,
            patch_size=patch_size,
            in_channels=in_chans
        )
        self.enc_dims = [96, 192, 384, 768]
        
        # Decoder
        self.decoder = nn.ModuleList([
            nn.ConvTranspose2d(self.enc_dims[3], self.enc_dims[2], kernel_size=2, stride=2),
            nn.Conv2d(self.enc_dims[2] * 2, self.enc_dims[2], kernel_size=3, padding=1),
            nn.ConvTranspose2d(self.enc_dims[2], self.enc_dims[1], kernel_size=2, stride=2),
            nn.Conv2d(self.enc_dims[1] * 2, self.enc_dims[1], kernel_size=3, padding=1),
            nn.ConvTranspose2d(self.enc_dims[1], self.enc_dims[0], kernel_size=2, stride=2),
            nn.Conv2d(self.enc_dims[0] * 2, self.enc_dims[0], kernel_size=3, padding=1),
            nn.Conv2d(self.enc_dims[0], out_chans, kernel_size=3, padding=1)
        ])
        
        # Text Reconstruction Module (TRM)
        self.trm = nn.Sequential(
            nn.Conv2d(self.enc_dims[0], 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, out_chans, kernel_size=3, padding=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        enc_features = []
        x = self.swin(pixel_values=x).hidden_states
        enc_features = x[1:]
        
        x = enc_features[-1]
        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x)
            skip_idx = len(enc_features) - 2 - (i // 2)
            if skip_idx >= 0:
                x = torch.cat([x, enc_features[skip_idx]], dim=1)
            x = F.relu(self.decoder[i + 1](x))
        
        img_output = torch.tanh(x)
        text_output = self.trm(x)
        return img_output, text_output

In [11]:
# Discriminator with Partition Discriminator Module
class Discriminator(nn.Module):
    def __init__(self, img_size=(128, 256), in_chans=3, patch_size=32):
        super(Discriminator, self).__init__()
        self.global_disc = nn.Sequential(
            nn.Conv2d(in_chans, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=0)
        )
        
        self.patch_size = patch_size
        self.patch_disc = nn.Sequential(
            nn.Conv2d(in_chans, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1)
        )
    
    def forward(self, x):
        global_out = self.global_disc(x)
        b, c, h, w = x.shape
        patches = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        patches = patches.contiguous().view(b, c, -1, self.patch_size, self.patch_size)
        patch_out = []
        for i in range(patches.size(2)):
            patch = patches[:, :, i, :, :]
            patch_out.append(self.patch_disc(patch))
        patch_out = torch.stack(patch_out, dim=2)
        return global_out, patch_out

In [13]:
# VGG Perceptual Loss
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        vgg = vgg19(pretrained=True).features
        self.vgg_layers = nn.Sequential(
            *list(vgg.children())[:16]
        ).eval()
        for param in self.vgg_layers.parameters():
            param.requires_grad = False
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
    def forward(self, x, y):
        x = self.normalize(x)
        y = self.normalize(y)
        x_vgg = self.vgg_layers(x)
        y_vgg = self.vgg_layers(y)
        return F.mse_loss(x_vgg, y_vgg)

In [ ]:
# Custom Dataset
class LicensePlateDataset(Dataset):
    def __init__(self, blur_dir, sharp_dir, transform=None):
        self.blur_files = sorted(os.listdir(blur_dir))
        self.sharp_files = sorted(os.listdir(sharp_dir))
        self.blur_dir = blur_dir
        self.sharp_dir = sharp_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.blur_files)
    
    def __getitem__(self, idx):
        blur_img = Image.open(os.path.join(self.blur_dir, self.blur_files[idx])).convert('RGB')
        sharp_img = Image.open(os.path.join(self.sharp_dir, self.sharp_files[idx])).convert('RGB')
        if self.transform:
            blur_img = self.transform(blur_img)
            sharp_img = self.transform(sharp_img)
        return blur_img, sharp_img

In [ ]:
# Data Preparation
from torchvision.transforms import Compose, Resize, ToTensor, RandomHorizontalFlip, ColorJitter

transform = Compose([
    Resize((128, 256)),
    ToTensor(),
    RandomHorizontalFlip(0.5),
    ColorJitter(brightness=0.2, contrast=0.2)
])

# Replace with your dataset paths
blur_dir = "path/to/blur_images"
sharp_dir = "path/to/sharp_images"

dataset = LicensePlateDataset(blur_dir=blur_dir, sharp_dir=sharp_dir, transform=transform)

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} pairs, Val: {len(val_dataset)} pairs, Test: {len(test_dataset)} pairs")

In [ ]:
# Gradient Penalty for WGAN-GP
def compute_gradient_penalty(discriminator, real_samples, fake_samples, device):
    alpha = torch.rand(real_samples.size(0), 1, 1, 1, device=device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    d_global, d_patches = discriminator(interpolates)
    fake = torch.ones(d_global.size(), device=device)
    
    gradients = torch.autograd.grad(
        outputs=(d_global, d_patches.mean()),
        inputs=interpolates,
        grad_outputs=(fake, fake),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

In [ ]:
# Training Loop
def train_model(generator, discriminator, train_loader, val_loader, device, num_epochs=50):
    g_optimizer = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    perceptual_loss = VGGPerceptualLoss().to(device)
    criterion = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler()
    
    train_losses = {'g_loss': [], 'd_loss': []}
    val_losses = []
    
    for epoch in range(num_epochs):
        generator.train()
        discriminator.train()
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        
        for i, (blur_imgs, sharp_imgs) in enumerate(train_loader):
            blur_imgs, sharp_imgs = blur_imgs.to(device), sharp_imgs.to(device)
            
            # Train Discriminator
            with torch.cuda.amp.autocast():
                d_optimizer.zero_grad()
                real_global, real_patches = discriminator(sharp_imgs)
                fake_img, fake_text = generator(blur_imgs)
                fake_global, fake_patches = discriminator(fake_img.detach())
                
                d_loss_real = -torch.mean(real_global) - torch.mean(real_patches)
                d_loss_fake = torch.mean(fake_global) + torch.mean(fake_patches)
                gp = compute_gradient_penalty(discriminator, sharp_imgs, fake_img, device)
                d_loss = d_loss_real + d_loss_fake + 10.0 * gp
            
            scaler.scale(d_loss).backward()
            scaler.step(d_optimizer)
            scaler.update()
            
            # Train Generator
            with torch.cuda.amp.autocast():
                g_optimizer.zero_grad()
                fake_img, fake_text = generator(blur_imgs)
                fake_global, fake_patches = discriminator(fake_img)
                
                adv_loss = -torch.mean(fake_global) - torch.mean(fake_patches)
                perc_loss = perceptual_loss(fake_img, sharp_imgs)
                text_loss = criterion(fake_text, sharp_imgs)
                g_loss = adv_loss + 100.0 * perc_loss + 10.0 * text_loss
            
            scaler.scale(g_loss).backward()
            scaler.step(g_optimizer)
            scaler.update()
            
            epoch_g_loss += g_loss.item()
            epoch_d_loss += d_loss.item()
            
            if i % 100 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] Batch [{i}/{len(train_loader)}] "
                      f"D Loss: {d_loss.item():.4f} G Loss: {g_loss.item():.4f}")
        
        # Validation
        generator.eval()
        val_loss = 0.0
        with torch.no_grad():
            for blur_imgs, sharp_imgs in val_loader:
                blur_imgs, sharp_imgs = blur_imgs.to(device), sharp_imgs.to(device)
                with torch.cuda.amp.autocast():
                    fake_img, fake_text = generator(blur_imgs)
                    perc_loss = perceptual_loss(fake_img, sharp_imgs)
                    text_loss = criterion(fake_text, sharp_imgs)
                    val_loss += (100.0 * perc_loss + 10.0 * text_loss).item()
        val_loss /= len(val_loader)
        
        train_losses['g_loss'].append(epoch_g_loss / len(train_loader))
        train_losses['d_loss'].append(epoch_d_loss / len(train_loader))
        val_losses.append(val_loss)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] Val Loss: {val_loss:.4f}")
        
        # Save checkpoint
        torch.save(generator.state_dict(), f"generator_epoch_{epoch+1}.pth")
        torch.save(discriminator.state_dict(), f"discriminator_epoch_{epoch+1}.pth")
    
    return train_losses, val_losses

In [ ]:
# Initialize and Train
generator = SwinUNet(img_size=(128, 256)).to(device)
discriminator = Discriminator(img_size=(128, 256)).to(device)

train_losses, val_losses = train_model(generator, discriminator, train_loader, val_loader, device, num_epochs=50)

In [ ]:
# Visualize Training Progress
plt.figure(figsize=(10, 5))
plt.plot(train_losses['g_loss'], label='Generator Loss')
plt.plot(train_losses['d_loss'], label='Discriminator Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Losses')
plt.show()

In [ ]:
# Test and Visualize Results
def visualize_results(generator, test_loader, device, num_samples=5):
    generator.eval()
    with torch.no_grad():
        for i, (blur_imgs, sharp_imgs) in enumerate(test_loader):
            if i >= num_samples:
                break
            blur_imgs, sharp_imgs = blur_imgs.to(device), sharp_imgs.to(device)
            fake_img, fake_text = generator(blur_imgs)
            
            # Convert to numpy for visualization
            blur_img = blur_imgs[0].cpu().permute(1, 2, 0).numpy()
            sharp_img = sharp_imgs[0].cpu().permute(1, 2, 0).numpy()
            fake_img = fake_img[0].cpu().permute(1, 2, 0).numpy()
            fake_text = fake_text[0].cpu().permute(1, 2, 0).numpy()
            
            plt.figure(figsize=(15, 3))
            plt.subplot(1, 4, 1)
            plt.imshow(blur_img)
            plt.title('Blurred')
            plt.axis('off')
            plt.subplot(1, 4, 2)
            plt.imshow(fake_img)
            plt.title('Deblurred')
            plt.axis('off')
            plt.subplot(1, 4, 3)
            plt.imshow(fake_text)
            plt.title('Text Enhanced')
            plt.axis('off')
            plt.subplot(1, 4, 4)
            plt.imshow(sharp_img)
            plt.title('Ground Truth')
            plt.axis('off')
            plt.show()

visualize_results(generator, test_loader, device)